# 情報通信実験第5 IoT 設計制作実験(Colab + AI)
手順は https://hasevr.github.io/ICTEx5/iot/ を見てください。

1. 上から順にセルを実行する(「Google Drive is disabled」は OK で閉じる)。
2. **前回の作業の続きなら**「前回の作業を戻す」で、前回ダウンロードした ZIP をアップロードする。
3. 「▶ ESP32 ブリッジを開く」を **Chrome か Edge の新しいタブ**で開き、ESP32 をつないで「ESP32 に接続」。
4. 左下の「ターミナル」で `cd /content/work && agy` として AI エージェントを起動する。
5. **終わる前に、必ず「作業を ZIP でダウンロード」を実行する**(Colab の VM はリセットされると中身が消えます)。

In [ ]:
%%bash
# 1. ESP-IDF v5.5.1 の準備
#    インストール済みの一式(GitHub のリリース。Python 本体・cmake・ninja も同梱)を取ってきて展開する。
#    Colab の Python の版や apt のパッケージには左右されない。万一動かなければ通常のインストール(5分ほど)
set -e
T0=$(date +%s)
REL=https://github.com/hasevr/ICTEx5/releases/download/colab-esp-idf-v5.5.1-r2
export IDF_TOOLS_PATH=/content/.espressif
cd /content
full_install() {
  echo "通常のインストールを行います(5分ほど)"
  PYV=$(/usr/bin/python3 -c 'import sys;print(f"{sys.version_info[0]}.{sys.version_info[1]}")')
  apt-get -qq update > /dev/null && apt-get -qq install -y flex bison gperf ninja-build ccache libffi-dev libssl-dev dfu-util libusb-1.0-0 python3-venv python${PYV}-venv > /dev/null
  rm -rf /content/esp-idf /content/.espressif
  git clone -q -b v5.5.1 --depth 1 --recursive --shallow-submodules https://github.com/espressif/esp-idf.git /content/esp-idf
  /content/esp-idf/install.sh esp32 > /content/idf-install.log 2>&1 || { tail -30 /content/idf-install.log; exit 1; }
  /usr/bin/python3 /content/esp-idf/tools/idf_tools.py install cmake ninja >> /content/idf-install.log 2>&1
}
if [ ! -d /content/esp-idf ]; then
  # 展開に使う zstd が無ければ入れる
  which zstd > /dev/null || apt-get -qq install -y zstd > /dev/null 2>&1 || { apt-get -qq update > /dev/null; apt-get -qq install -y zstd > /dev/null; }
  curl -sfL "$REL/esp-idf-v5.5.1-colab.tar.zst" | tar -C /content -I zstd -x || full_install
fi
# どのシェル(ターミナル・AI)からでも idf.py が使えるようにする(同梱の Python を優先。ccache は使わない)
cat > /usr/local/bin/idf.py <<'EOF'
#!/bin/bash
export IDF_TOOLS_PATH=/content/.espressif
export PATH="/content/.espressif/python/bin:$PATH"
export IDF_CCACHE_ENABLE=0
. /content/esp-idf/export.sh > /dev/null 2>&1
exec python3 "$IDF_PATH/tools/idf.py" "$@"
EOF
chmod +x /usr/local/bin/idf.py
# 取ってきたものが動かなければ、通常のインストールに切り替える
idf.py --version > /dev/null 2>&1 || { full_install; idf.py --version > /dev/null; }
echo "### ESP-IDF の準備: $(( $(date +%s) - T0 )) 秒($(idf.py --version))" 

In [ ]:
%%bash
# 2. 作業フォルダと、サンプル・AI への指示ファイルを用意する
set -e
mkdir -p /content/work && cd /content/work
if [ -d ictex5/.git ]; then git -C ictex5 pull -q --ff-only || echo "git pull できませんでした(手元の版で続けます)"
else git clone -q --depth 1 https://github.com/hasevr/ICTEx5 ictex5; fi
# AI への指示ファイル。自分たちのプロジェクトの説明は PROJECT.md に書く(AGENTS.md から読むよう指示してある)
(curl -sfL -o AGENTS.md https://raw.githubusercontent.com/hasevr/ICTEx5/main/agent/AGENTS.md && cp AGENTS.md GEMINI.md) || echo "AGENTS.md を取得できませんでした"
[ -f PROJECT.md ] || printf '# 私たちの IoT システム\n\n(構想、この人が担当する部分、ピンの割り当て、通信の仕様をここに書く)\n' > PROJECT.md
cat > /usr/local/bin/xdg-open <<'EOF'
#!/bin/sh
echo "$1" > /content/agy_url.txt
EOF
chmod +x /usr/local/bin/xdg-open
echo "サンプル: ictex5/firmware/ の HttpHaptic(無線LAN + HTTP)、Mpu9250(9 軸センサ)、I2SHaptic(D 級アンプ)、ictex5/server/app.py(Raspberry Pi のサーバ)"
ls /content/work

## 前回の作業を戻す(2回目以降)
前回の最後にダウンロードした `iot-work-....zip` を選ぶ。

In [ ]:
from google.colab import files
import zipfile, io
up = files.upload()
for name, data in up.items():
    zipfile.ZipFile(io.BytesIO(data)).extractall('/content/work')
    print(name, 'を /content/work に戻しました')
!ls /content/work

## 新しいプロジェクトを作る(1回目)
サンプルをコピーして始めるのが簡単です(名前は自由に。下の例は `myiot`)。AI に「HttpHaptic をもとに `myiot` を作って」と頼んでもかまいません。

In [ ]:
!test -d /content/work/myiot || cp -r /content/work/ictex5/firmware/HttpHaptic /content/work/myiot; ls /content/work/myiot
!idf.py -C /content/work/myiot build 2>&1 | tail -3

In [ ]:
# 4. ESP32 ブリッジを起動する(リンクは Chrome / Edge の新しいタブで開く)
!rm -rf /content/esp-bridge && git clone -q https://github.com/hasevr/esp-bridge /content/esp-bridge
import sys; sys.path.insert(0, '/content/esp-bridge')
import esp_bridge; esp_bridge.start()

## AI エージェント(Antigravity CLI)
左下の「ターミナル」で次を実行し、**1. Google OAuth** を選ぶ。
```
cd /content/work && agy
```
ログイン用の URL が出たら、下のセルを実行して出るリンクから大学アカウントでログインし、表示されたコードをターミナルに貼り付ける。
「Terms of Service & Data Use」では、改善への協力のチェックを**スペースで外して** Done。

In [ ]:
# ログイン用リンク(agy でログイン方法を選んだ後に実行)
import os
from IPython.display import HTML
u = open('/content/agy_url.txt').read().strip() if os.path.exists('/content/agy_url.txt') else ''
HTML(f'<a href="{u}" target="_blank" style="font-size:18px">▶ Antigravity にログイン</a>' if u else 'まだ URL がありません。ターミナルで agy を起動し、1. Google OAuth を選んでから実行してください。')

## 手で書き込む・出力を見る(AI を使わない場合)

In [ ]:
!esp flash /content/work/myiot
!esp log --since-flash -n 30

## 作業を ZIP でダウンロードする(毎回、終わる前に必ず)
`/content/work` の中の、自分たちのファイル(ビルドの結果やサンプルの複製を除く)をまとめてダウンロードします。
班のメンバーと共有するときも、この ZIP を渡します。

In [ ]:
import os, zipfile, time
from google.colab import files
skip_dirs = {'build', 'managed_components', 'ictex5', '.git', '__pycache__'}
name = time.strftime('iot-work-%Y%m%d-%H%M.zip')
with zipfile.ZipFile('/content/' + name, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, dirs, fs in os.walk('/content/work'):
        dirs[:] = [d for d in dirs if d not in skip_dirs]
        for f in fs:
            if f in ('sdkconfig.old', 'GEMINI.md'):
                continue
            p = os.path.join(root, f)
            z.write(p, os.path.relpath(p, '/content/work'))
print(name, os.path.getsize('/content/' + name) // 1024, 'KB')
files.download('/content/' + name)

## Raspberry Pi に送るファイルをダウンロードする
例: サーバのプログラム。PC で `scp app.py pi@<Raspberry Pi の IP アドレス>:` として送る。

In [ ]:
from google.colab import files
files.download('/content/work/ictex5/server/app.py')   # 自分たちのファイルに書き換える

## AI 利用の記録
主な依頼・AI の変更・うまくいかなかったこと・測った結果を、ここに書いていく(個人レポートの材料)。

| 日付・時刻 | 依頼したこと | AI がしたこと | 結果・気づき |
|---|---|---|---|
| | | | |